In [87]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor , export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import roc_auc_score , root_mean_squared_error
import xgboost as xgb
import seaborn as sns
from matplotlib import pyplot as plt

In [2]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

--2026-10-29 17:25:33--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635180 (620K) [text/plain]
Saving to: ‘car_fuel_efficiency_2026.csv’

car_fuel_efficiency 100%[===================>] 620.29K  --.-KB/s    in 0.02s   

2026-10-29 17:25:34 (37.6 MB/s) - ‘car_fuel_efficiency_2026.csv’ saved [635180/635180]



In [40]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')
df.isnull().sum()
df.fillna(0, inplace=True)
df.isnull().sum()

,0
model_year,0
origin,0
fuel_type,0
drivetrain,0
num_doors,0
engine_displacement,0
num_cylinders,0
horsepower,0
vehicle_weight,0
acceleration,0


In [45]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

dv = DictVectorizer(sparse=False)

train_dict = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val.to_dict(orient='records')
X_val = dv.transform(val_dict)

test_dict = df_test.to_dict(orient='records')
X_test = dv.transform(test_dict)

In [47]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_val)

export_text(dt, feature_names=dv.feature_names_)

'|--- model_year <= 1997.50\n|   |--- value: [28.61]\n|--- model_year >  1997.50\n|   |--- value: [31.17]\n'

In [50]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
round(rmse, 4)

1.8392

In [59]:
for d in [10, 15, 20, 25]:
    for n in [10, 50, 100, 150]:
      rf = RandomForestRegressor(n_estimators=n,max_depth=d, random_state=1, n_jobs=-1)
      rf.fit(X_train, y_train)
      y_pred = rf.predict(X_val)
      rmse = root_mean_squared_error(y_val, y_pred)
      print(d, "        ", n ,"      ",  round(rmse, 4))
    print(f'\n')

10          10        1.7864
10          50        1.7483
10          100        1.7451
10          150        1.744


15          10        1.8367
15          50        1.7724
15          100        1.7692
15          150        1.7678


20          10        1.8412
20          50        1.7726
20          100        1.768
20          150        1.7672


25          10        1.8395
25          50        1.7744
25          100        1.7695
25          150        1.7689




In [61]:
#the best: 10          150        1.744

In [62]:
rf = RandomForestRegressor(n_estimators=10,max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)

In [73]:
feature_importance = zip(dv.feature_names_, rf.feature_importances_)
feature_importance = sorted(feature_importance, key=lambda x: x[1], reverse=True)


for feature, importance in feature_importance:
  print(f'{feature}: {importance}')

model_year: 0.29591682444495776
vehicle_weight: 0.2034201442933393
fuel_type=Gasoline: 0.16071252496131586
horsepower: 0.0777318489689458
engine_displacement: 0.06168228356565586
acceleration: 0.05538692324902256
origin=USA: 0.04229853480078026
drivetrain=Front-wheel drive: 0.020116426827607482
fuel_type=Diesel: 0.020036810463992277
num_doors: 0.01932199321848706
fuel_type=Hybrid: 0.012820989447342824
drivetrain=All-wheel drive: 0.007805375661003059
num_cylinders: 0.007329473845400125
origin=Asia: 0.006651691870276565
drivetrain=Rear-wheel drive: 0.005527393936894499
origin=Europe: 0.0032407604449787124


In [90]:
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=dv.feature_names_)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=dv.feature_names_)
model = xgb.train(xgb_params, dtrain, num_boost_round=100)
y_pred = model.predict(dval)
rmse = root_mean_squared_error(y_val, y_pred)
rmse

1.8264579799904583

In [91]:
xgb_params = {
    'eta': 0.1,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=dv.feature_names_)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=dv.feature_names_)
model = xgb.train(xgb_params, dtrain, num_boost_round=100)
y_pred = model.predict(dval)
rmse = root_mean_squared_error(y_val, y_pred)
rmse

1.724810059827976